In [57]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, log_loss, classification_report
import joblib
import os

# 1. Load the unified dataset
df = pd.read_csv('../data/processed/epl_model_features.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# 2. Define the Target Class
# 2 = Home Win, 1 = Draw, 0 = Away Win
conditions = [
    df['FTHG'] > df['FTAG'],
    df['FTHG'] == df['FTAG'],
    df['FTHG'] < df['FTAG']
]
df['Target'] = np.select(conditions, [2, 1, 0], default=1)

In [58]:
# --- A. ELO Ratings ---
df['Elo_Diff'] = df['Home_Elo'] - df['Away_Elo']

# --- B. Multi-Scale Form (3, 5, and 10 games) ---
for w in [3, 5, 10]:
    # Attack vs Defense (Home Attack vs Away Defense, and vice versa)
    df[f'xG_Attack_Diff_roll{w}'] = df[f'Home_xG_Created_roll{w}'] - df[f'Away_xG_Conceded_roll{w}']
    df[f'xG_Defense_Diff_roll{w}'] = df[f'Away_xG_Created_roll{w}'] - df[f'Home_xG_Conceded_roll{w}']
    
    # Match Context / Pressure metrics
    df[f'Corner_Diff_roll{w}'] = df[f'Home_Corners_roll{w}'] - df[f'Away_Corners_roll{w}']
    df[f'Foul_Diff_roll{w}'] = df[f'Home_Fouls_roll{w}'] - df[f'Away_Fouls_roll{w}']

# --- C. Venue-Specific Form ---
# Comparing Home team's home form vs Away team's away form
df['Venue_xG_Attack_Diff'] = df['Home_xG_Created_Venue_roll5'] - df['Away_xG_Conceded_Venue_roll5']
df['Expected_Match_xG'] = df['Home_xG_Created_roll5'] + df['Away_xG_Created_roll5']

# --- D. Schedule Context & Fatigue ---
df['Rest_Diff'] = df['Home_Rest_Days'] - df['Away_Rest_Days']
df['Congestion_Diff'] = df['Home_Congestion_Flag'] - df['Away_Congestion_Flag']

# --- E. Market Baseline (Odds to Implied Probabilities) ---
raw_margin = (1 / df['B365H']) + (1 / df['B365D']) + (1 / df['B365A'])
df['Bookie_Prob_H'] = (1 / df['B365H']) / raw_margin
df['Bookie_Prob_D'] = (1 / df['B365D']) / raw_margin
df['Bookie_Prob_A'] = (1 / df['B365A']) / raw_margin

In [59]:
feature_cols = [
    # 1. Team Quality Baseline
    'Elo_Diff', 'Home_Elo', 'Away_Elo',
    
    # 2. Multi-Scale Form
    'xG_Attack_Diff_roll3', 'xG_Defense_Diff_roll3',
    'xG_Attack_Diff_roll5', 'xG_Defense_Diff_roll5',
    'xG_Attack_Diff_roll10', 'xG_Defense_Diff_roll10',
    'Corner_Diff_roll5', 'Foul_Diff_roll5',
    
    # 3. Venue & Match Dynamics
    'Venue_xG_Attack_Diff', 'Expected_Match_xG',
    
    # 4. Schedule/Fatigue
    'Rest_Diff', 'Congestion_Diff',
    
    # 5. Market Anchor
    'Bookie_Prob_H', 'Bookie_Prob_D', 'Bookie_Prob_A'
]

X = df[feature_cols]
y = df['Target']

# 80/20 Chronological Split
split_idx = int(len(df) * 0.80)
X_train, y_train = X.iloc[:split_idx], y.iloc[:split_idx]
X_test, y_test = X.iloc[split_idx:], y.iloc[split_idx:]

print(f"Training on {len(X_train)} matches, testing on {len(X_test)} matches.")

Training on 1163 matches, testing on 291 matches.


In [115]:
# Regularized base XGBoost
base_xgb = XGBClassifier(
    n_estimators=120,
    max_depth=2,
    learning_rate=0.015,
    reg_alpha=1.5,
    reg_lambda=2.5,
    subsample=0.75,
    colsample_bytree=0.75,
    objective='multi:softprob',
    num_class=3,
    random_state=42
)

# Calibrate probabilities across 3 internal folds
calibrated_model = CalibratedClassifierCV(
    estimator=base_xgb,
    method='isotonic',
    cv=3
)

print("Training model...")
calibrated_model.fit(X_train, y_train)
print("Training complete!")

Training model...
Training complete!


In [116]:
# Predict probabilities on the unseen test set
y_probs = calibrated_model.predict_proba(X_test)

# Custom decision threshold to force the model to predict more draws
def custom_match_predictor(probs, draw_threshold=0.285):
    preds = []
    for row in probs:
        p_away, p_draw, p_home = row[0], row[1], row[2]
        if p_draw >= draw_threshold:
            preds.append(1)  # Draw
        elif p_home > p_away:
            preds.append(2)  # Home Win
        else:
            preds.append(0)  # Away Win
    return np.array(preds)

y_preds = custom_match_predictor(y_probs, draw_threshold=0.285)

# Calculate Evaluation Metrics
model_loss = log_loss(y_test, y_probs)
model_acc = accuracy_score(y_test, y_preds)

bookie_probs = X_test[['Bookie_Prob_A', 'Bookie_Prob_D', 'Bookie_Prob_H']].values
bookie_loss = log_loss(y_test, bookie_probs)
bookie_acc = accuracy_score(y_test, np.argmax(bookie_probs, axis=1))

print("\n=== MODEL VS BOOKMAKER ===")
print(f"Model Accuracy:  {model_acc:.1%} | Model Log-Loss:  {model_loss:.4f}")
print(f"Bookie Accuracy: {bookie_acc:.1%} | Bookie Log-Loss: {bookie_loss:.4f}\n")

print(classification_report(y_test, y_preds, target_names=['Away Win', 'Draw', 'Home Win'], zero_division=0))

# Save the finished model
os.makedirs('../models', exist_ok=True)
joblib.dump(calibrated_model, '../models/calibrated_xgb_outcome.pkl')
print("Model saved to ../models/calibrated_xgb_outcome.pkl")


=== MODEL VS BOOKMAKER ===
Model Accuracy:  47.1% | Model Log-Loss:  1.0267
Bookie Accuracy: 49.1% | Bookie Log-Loss: 1.0260

              precision    recall  f1-score   support

    Away Win       0.45      0.47      0.46        88
        Draw       0.24      0.06      0.09        85
    Home Win       0.51      0.77      0.61       118

    accuracy                           0.47       291
   macro avg       0.40      0.43      0.39       291
weighted avg       0.41      0.47      0.41       291

Model saved to ../models/calibrated_xgb_outcome.pkl


In [122]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from scipy.stats import poisson
import joblib

def fit_dixon_coles(df_raw, xi=0.0032):
    """
    Fits Dixon-Coles parameters using Maximum Likelihood Estimation.
    xi=0.0032 corresponds to a half-life of approximately 216 days.
    """
    print("Extracting teams and applying time decay weighting...")
    # Filter to necessary columns and ensure chronological order
    df = df_raw[['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG']].copy()
    
    teams = np.sort(np.unique(df[['HomeTeam', 'AwayTeam']].values))
    n_teams = len(teams)
    team_to_idx = {team: i for i, team in enumerate(teams)}
    
    df['Home_Idx'] = df['HomeTeam'].map(team_to_idx)
    df['Away_Idx'] = df['AwayTeam'].map(team_to_idx)
    
    # Calculate time decay weights: phi(t) = exp(-xi * t)
    max_date = df['Date'].max()
    df['Days_Ago'] = (max_date - df['Date']).dt.days
    weights = np.exp(-xi * df['Days_Ago'].values)
    
    # Negative Log-Likelihood Objective Function
    def log_likelihood(params):
        alpha = params[:n_teams]          # Attack parameters
        beta = params[n_teams:2*n_teams]  # Defense parameters
        gamma = params[-1]                # Global Home Advantage
        
        # Calculate expected goals (lambda, mu)
        lam = alpha[df['Home_Idx']] * beta[df['Away_Idx']] * gamma
        mu = alpha[df['Away_Idx']] * beta[df['Home_Idx']]
        
        # Poisson log probability: x*ln(lam) - lam (constant log(x!) is dropped)
        llk_home = df['FTHG'] * np.log(lam) - lam
        llk_away = df['FTAG'] * np.log(mu) - mu
        
        # Return negative sum for minimization
        return -np.sum(weights * (llk_home + llk_away))
        
    # Initial Guesses & Bounds
    init_params = np.ones(2 * n_teams + 1)
    init_params[-1] = 1.25 # Initial home advantage guess
    
    bounds = [(0.01, 5.0)] * (2 * n_teams) + [(0.5, 3.0)]
    
    # Identifiability Constraint: Average attack parameter must equal 1.0
    constraints = [{'type': 'eq', 'fun': lambda x: sum(x[:n_teams]) - n_teams}]
    
    print("Optimizing parameters via SLSQP... (This may take a moment)")
    opt_res = minimize(
        log_likelihood, 
        init_params, 
        bounds=bounds, 
        constraints=constraints, 
        method='SLSQP',
        options={'maxiter': 200}
    )
    
    # Package optimized parameters into dictionaries
    dc_params = {
        'attack': {team: opt_res.x[i] for team, i in team_to_idx.items()},
        'defense': {team: opt_res.x[n_teams + i] for team, i in team_to_idx.items()},
        'home_adv': opt_res.x[-1]
    }
    
    # Save the parameters for live inference
    joblib.dump(dc_params, '../models/dc_mle_params.pkl')
    print(rf"MLE Fit Complete! Global Home Advantage ($\gamma$): {dc_params['home_adv']:.3f}")
    
    return dc_params

In [118]:
def extract_top_scorelines(home_xg, away_xg, max_goals=6):
    """
    Generates a Poisson matrix up to max_goals x max_goals,
    returning the top 3 formatted scoreline predictions.
    """
    # 1. Generate probabilities for 0 to 5 goals
    home_probs = poisson.pmf(np.arange(max_goals), home_xg)
    away_probs = poisson.pmf(np.arange(max_goals), away_xg)
    
    # 2. Outer product creates the independent 2D probability grid
    grid = np.outer(home_probs, away_probs)
    
    # 3. Flatten into a sortable list
    scorelines = []
    for h in range(max_goals):
        for a in range(max_goals):
            scorelines.append({
                'score': f"{h}-{a}", 
                'prob': grid[h][a]
            })
            
    df_scores = pd.DataFrame(scorelines).sort_values(by='prob', ascending=False)
    
    # 4. Extract Top 3 with formatted percentages
    top_3 = df_scores.head(3).copy()
    top_3['formatted'] = top_3.apply(
        lambda row: f"{row['score']} ({row['prob']:.1%})", 
        axis=1
    )
    
    return top_3['formatted'].tolist()

In [124]:
def generate_match_forecast(home_team, away_team, xgb_model, dc_params, df_features, feature_cols=None):
    """
    Unified inference pipeline utilizing both the Phase 4 Machine Learning 
    Classifier and the Dixon-Coles MLE Scoreline fit.
    """
    if feature_cols is None:
        feature_cols = [
            'Elo_Diff', 'Home_Elo', 'Away_Elo',
            'xG_Attack_Diff_roll3', 'xG_Defense_Diff_roll3',
            'xG_Attack_Diff_roll5', 'xG_Defense_Diff_roll5',
            'xG_Attack_Diff_roll10', 'xG_Defense_Diff_roll10',
            'Corner_Diff_roll5', 'Foul_Diff_roll5',
            'Venue_xG_Attack_Diff', 'Expected_Match_xG',
            'Rest_Diff', 'Congestion_Diff',
            'Bookie_Prob_H', 'Bookie_Prob_D', 'Bookie_Prob_A'
        ]

    print(f"\n{'='*45}")
    print(f" MATCH FORECAST: {home_team} vs {away_team}")
    print(f"{'='*45}")
    
    # --- 1. Dixon-Coles Exact Scoreline Generation ---
    alpha_h = dc_params['attack'].get(home_team, 1.0)
    beta_h  = dc_params['defense'].get(home_team, 1.0)
    alpha_a = dc_params['attack'].get(away_team, 1.0)
    beta_a  = dc_params['defense'].get(away_team, 1.0)
    gamma   = dc_params['home_adv']
    
    dc_home_xg = alpha_h * beta_a * gamma
    dc_away_xg = alpha_a * beta_h
    
    top_scores = extract_top_scorelines(dc_home_xg, dc_away_xg)
    
    print("\n[ EXACT SCORELINE PROBABILITIES ]")
    print(f"  Projected xG: {home_team} ({dc_home_xg:.2f}) - {away_team} ({dc_away_xg:.2f})")
    print(f"  1. {top_scores[0]}")
    print(f"  2. {top_scores[1]}")
    print(f"  3. {top_scores[2]}")
    
    # --- 2. XGBoost Match Outcome Generation ---
    match_vector = df_features[
        (df_features['HomeTeam'] == home_team) & 
        (df_features['AwayTeam'] == away_team)
    ].iloc[-1:] 
    
    # Select only the exact feature columns used during model training
    X_live = match_vector[feature_cols]
    
    # Execute inference
    probs = xgb_model.predict_proba(X_live)[0]
    
    print("\n[ MATCH OUTCOME CLASSIFICATION ]")
    print(f"  {home_team} Win : {probs[2]:.1%}")
    print(f"  Draw      : {probs[1]:.1%}")
    print(f"  {away_team} Win : {probs[0]:.1%}")
    print(f"{'='*45}\n")

In [150]:
# Execution Example:
dc_params = fit_dixon_coles(df_raw, xi=0.0032)
generate_match_forecast("Man City", "Bournemouth", calibrated_model, dc_params, df)

Extracting teams and applying time decay weighting...
Optimizing parameters via SLSQP... (This may take a moment)
MLE Fit Complete! Global Home Advantage ($\gamma$): 1.205

 MATCH FORECAST: Man City vs Bournemouth

[ EXACT SCORELINE PROBABILITIES ]
  Projected xG: Man City (2.08) - Bournemouth (0.94)
  1. 2-0 (10.6%)
  2. 1-0 (10.2%)
  3. 2-1 (9.9%)

[ MATCH OUTCOME CLASSIFICATION ]
  Man City Win : 70.1%
  Draw      : 17.3%
  Bournemouth Win : 12.6%

